#####   行为数据预处理 Pipeline：将实验硬件event文件（.evt）解析并修复后输出标准化的 .pkl 文件

内核：base

In [ ]:
dir = '/mnt/HDD1/zyyk78/Stage4/E01' #必须要细化到鼠、
#home/zyyk78/HDD/zyyk78/（软连接快捷方式）
log_dir='/mnt/HDD1/zyyk78/llom_log'

准备好待处理文件夹后，直接运行下列代码即可


In [44]:
#初始化
import os
import fixing
from ParseEvent import ParseEvent
import pickle
import pandas as pd
import warnings
import re
import traceback
warnings.simplefilter(action='ignore', category=FutureWarning)

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

def find_evt_files_without_pkl(dir=str):
    evt_files_without_pkl = []
    for root, dirs, files in os.walk(dir):
        for file in files:
            if file.endswith('.evt'):
                pkl_file_name = os.path.splitext(file)[0] + '.pkl'
                if pkl_file_name not in files:
                    evt_file_path = os.path.join(root, file)
                    evt_files_without_pkl.append(evt_file_path)
    
    return evt_files_without_pkl

Paths = find_evt_files_without_pkl(dir)
count=0
Event=None
if(len(Paths)==0):
    print('No evt to be processed!\n')
else:
    print('Find '+str(len(Paths))+' File(s)!')

delta1=25000
delta2=10000
df = pd.DataFrame()
if os.path.exists(os.path.join(dir,"README.xlsx")):
    df = pd.read_excel(os.path.join(dir,"README.xlsx"),sheet_name='Sheet1',header=None) .iloc[2:,:]
    print(f"df[0] dtype: {df[0].dtype}")
    # 读取 README 并将日期列转为 int
    df[0] = pd.to_numeric(df[0], errors='coerce').astype('Int64')  # 转为 Int64
    print('Find CSV data!')
    print(f"df[0] dtype: {df[0].dtype}")
else:
    print('No CSV data!')



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Find 13 File(s)!
df[0] dtype: object
Find CSV data!
df[0] dtype: Int64


Process All File Automatically！

In [45]:
while (count < len(Paths)):
    print('Processing file '+str(count+1)+'/'+str(len(Paths))+': '+Paths[count])
    try:
        Event=ParseEvent(Paths[count])
    except Exception as e:
        print('An error occoure while processing Event!' , e)
        Event = {
            'SlnL':{ 'Time_S':[],'Time_E':[]},
            'SlnR':{ 'Time_S':[],'Time_E':[]},
            'ROmL':{ 'Time_S':[],'Time_E':[]},
            'ROmR':{ 'Time_S':[],'Time_E':[]},
            'ModL':{ 'Time_S':[],'Time_E':[]},
            'ModR':{ 'Time_S':[],'Time_E':[]},
            'LickL':{ 'Time_S':[],'Time_E':[]},
            'LickR':{ 'Time_S':[],'Time_E':[]}
        }
    pattern = r'\d{8}'
    target_date = re.findall(pattern, os.path.basename(Paths[count]).split('_')[0]) [0]
    log_file = os.path.join(log_dir, f"log{target_date}.txt")
    matched_row=pd.DataFrame

    if not df.empty:
        matched_row = df[df[0]==int(target_date)]
        if len(matched_row) > 0:
            print('Find matched row in CSV data!')
            print(f'Target Tring mode: {matched_row.iloc[0,2]}')
        else:
            print(f'⚠️ WARNING: No matching row in README.xlsx for date {target_date}. Please add the record to README!')
    try:
        Event=fixing.fixing(Event,26000,39000,log_file,matched_row)
    except Exception as e:
        print(f'Error occurred while fixing the event: {e}')
        traceback.print_exc()
        Event=None
    if (Event != None):
        pkl_path=os.path.splitext(Paths[count])[0] + '.pkl'
        with open(pkl_path,"wb") as pkl_f:
            pickle.dump(Event,pkl_f)
            Event=None
        print('File Saved to: '+pkl_path)
    else:
        print('Event = Null!! Pkl won\'t be saved!')
    count += 1
print('All Finished!\n')

Processing file 1/13: /mnt/HDD1/zyyk78/Stage4/E01/20260403_P9F_00.evt
Find matched row in CSV data!
Target Tring mode: P9F
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Log file /mnt/HDD1/zyyk78/llom_log/log20260403.txt found. Event is matched to 7, time difference is 6 seconds.
No ModR Event! Fixing 
Fixing with log! 
Matching SlnL
181 events in hw, 181 events in sw, 178 matched!
Fit data: a=1.00, b=-12.66(s), r^2=1.00
Matching SlnR
336 events in hw, 335 events in sw, 333 matched!
Fit data: a=1.00, b=-12.77(s), r^2=1.00
ROmL Event too short! Removed!
Matching ROmR
43 events in hw, 42 events in sw, 41 matched!
Fit data: a=1.00, b=-13.09(s), r^2=1.00
3 Events matched! a=1.00±0.00, b=-12.84±0.18(s)
No ModR Event! Fixing 
File Saved to: /mnt/HDD1/zyyk78/Stage4/E01/20260403_P9F_00.pkl
Processing file 2/13: /mnt/HDD1/zyyk78/Stage4/E01/20260415_T9F_00.evt
Find matched row in CSV data!
Target Tring mode: T9F
R

/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid valu

Find matched row in CSV data!
Target Tring mode: P9F
No missing event, skipping fixing
File Saved to: /mnt/HDD1/zyyk78/Stage4/E01/20260402_P9F_00.pkl
Processing file 13/13: /mnt/HDD1/zyyk78/Stage4/E01/20260410_T9F_00.evt
Find matched row in CSV data!
Target Tring mode: T9F
No missing event, skipping fixing
File Saved to: /mnt/HDD1/zyyk78/Stage4/E01/20260410_T9F_00.pkl
All Finished!



#####   补充功能一：补全缺失起止时间的Readme
根据evt文件的修改时间，匹配对应llom_log日志中的起止时间

In [5]:
from pathlib import Path
from datetime import datetime, timedelta
import re

# 输入指定目录，以及目标日期区间
# -------------------------------------------------------------------------------

path = Path("/mnt/HDD1/zyyk78/MPtest/B05")
start_date = datetime.strptime("20250725", "%Y%m%d")
end_date = datetime.strptime("20250727", "%Y%m%d")

# -------------------------------------------------------------------------------

log_path = Path("/mnt/HDD1/zyyk78/llom_log")
# 获取所有 EVT 文件及其修改时间
evt_files = list(path.rglob("*.evt"))
files_with_mtime = []
for file in evt_files:
    if len(file.name) >= 11 and file.name[:8].isdigit():
        date_str = file.name[:8]
        try:
            file_date = datetime.strptime(date_str, "%Y%m%d")
            if start_date <= file_date <= end_date:
                stat_info = file.stat()
                mtime = datetime.fromtimestamp(stat_info.st_mtime)
                files_with_mtime.append((mtime, file))
        except ValueError:
            pass

files_with_mtime.sort(key=lambda x: x[0])

print(f"当前指定目录为: {path}")

for mtime, evt_file in files_with_mtime:
    # 获取日期对应的 log 文件
    date_str = evt_file.name[:8]
    log_file = log_path / f"log{date_str}.txt"
    
    if not log_file.exists():
        print(f"No log file for {date_str}")
        continue
    
    # 解析 log 文件中的所有 start 和 r 行
    log_lines = []
    with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line_lower = line.lower()
            if '->start' in line_lower or '->trial start' in line_lower or '<-r' in line_lower:
                match = re.match(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\.\d+', line)
                if match:
                    log_time = datetime.strptime(match.group(1), "%Y-%m-%d %H:%M:%S")
                    log_lines.append((log_time, line.rstrip()))
    
    # 找到最接近 EVT mtime 的 r 行（在10分钟误差内）
    target_time = mtime.replace(second=mtime.second, microsecond=0)
    time_diff = timedelta(minutes=10)
    matched_r = None
    
    for log_time, log_line in log_lines:
        if '<-r' in log_line.lower():
            diff = abs((log_time - target_time).total_seconds())
            if diff < time_diff.total_seconds():
                time_diff = timedelta(seconds=diff)
                matched_r = (log_time, log_line)
    
    if matched_r:
        log_time, r_line = matched_r
        # 找到 r 行的索引
        r_index = None
        for i, (lt, ll) in enumerate(log_lines):
            if lt == log_time and '<-r' in ll.lower():
                r_index = i
                break
        
        if r_index is not None:
            # 找到上一个 r 的位置
            prev_r_index = None
            for i in range(r_index - 1, -1, -1):
                if '<-r' in log_lines[i][1].lower():
                    prev_r_index = i
                    break
            
            start_idx = prev_r_index + 1 if prev_r_index is not None else 0
            
            # 获取开始时间和结束时间
            start_time = log_lines[start_idx][0]
            end_time = log_lines[r_index][0]
            duration = end_time - start_time
            
            # 格式化输出
            evt_name = evt_file.name.replace('.evt', '')
            start_str = start_time.strftime('%H:%M:%S')
            end_str = end_time.strftime('%H:%M:%S')
            duration_str = f"{int(duration.total_seconds()//3600):02d}:{int((duration.total_seconds()%3600)//60):02d}:{int(duration.total_seconds()%60):02d}"
            
            print(f"{evt_name}，开始时间：{start_str}，结束时间：{end_str}，用时{duration_str}")

当前指定目录为: /mnt/HDD1/zyyk78/MPtest/B05
No log file for 20250726


#####   觉得有不确定的地方，可以手动检查evt和txt日志文件：


1. 检索指定目录下、指定时间区间的evt文件及其文件修改时间(跑了就懂)

In [3]:
from pathlib import Path
from datetime import datetime

# -------------------------------------------------------------------------------
# path = "/home/zyyk78/HDD/zyyk78/Stage4"
# path = Path(path)
pkl_files = list(path.rglob("*.evt"))

# start = datetime.strptime("20260414", "%Y%m%d")
# end = datetime.strptime("20260414", "%Y%m%d")
# -------------------------------------------------------------------------------

files_with_mtime = []
for file in pkl_files:
    if len(file.name) >= 11 and file.name[:8].isdigit():
        date_str = file.name[:8]
        try:
            file_date = datetime.strptime(date_str, "%Y%m%d")
            if start <= file_date <= end:
                # Get file stats
                stat_info = file.stat()
                ctime = datetime.fromtimestamp(stat_info.st_ctime)
                mtime = datetime.fromtimestamp(stat_info.st_mtime)
                files_with_mtime.append((mtime, ctime, stat_info.st_size, file))
        except ValueError:
            pass

# Sort by modified time
files_with_mtime.sort(key=lambda x: x[0])

# Print in sorted order
for mtime, ctime, size, file in files_with_mtime:
    print(f"File: {file}")
    # print(f"  Created:  {ctime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Size: {size} bytes")
    print()

NameError: name 'start' is not defined

2.输出当天日志出现过的start和reset

In [48]:
from datetime import datetime, timedelta
from pathlib import Path

# -------------------------------------------------------------------------------

path = Path("/home/zyyk78/HDD/zyyk78/llom_log")
txt_files = list(path.rglob("*.txt"))

# start = datetime.strptime("20260414", "%Y%m%d")
# end = datetime.strptime("20260414", "%Y%m%d")

# -------------------------------------------------------------------------------

existing_files = {}
for file in txt_files:
    if file.name.startswith("log") and len(file.name) >= 11:
        date_str = file.name[3:11]
        try:
            file_date = datetime.strptime(date_str, "%Y%m%d")
            existing_files[file_date] = file
        except ValueError:
            pass

current = start
while current <= end:
    if current not in existing_files:
        current += timedelta(days=1)
        continue
    
    file = existing_files[current]
    print(f"\n=== {file} ===")
    skip_reset = False
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line_lower = line.lower()
            if 'start washing' in line_lower:
                skip_reset = True
            elif ('->start' in line_lower or "->trial start" in line_lower) and 'start washing' not in line_lower:
                skip_reset = False
            if ('->start' in line_lower or "->trial start" in line_lower) or '<-r' in line_lower:
                if not skip_reset:
                    print(line.rstrip())
    
    current += timedelta(days=1)


=== /home/zyyk78/HDD/zyyk78/llom_log/log20260414.txt ===
2026-04-14 12:18:55.791 +08:00 [INF] ->Trial start from left
2026-04-14 13:38:07.284 +08:00 [INF] <-r
2026-04-14 13:44:23.723 +08:00 [INF] ->Trial start from right
2026-04-14 14:12:59.234 +08:00 [INF] <-r
2026-04-14 14:20:19.953 +08:00 [INF] ->Trial start from right
2026-04-14 15:06:21.626 +08:00 [INF] <-r
2026-04-14 15:13:11.009 +08:00 [INF] ->Trial start from left
2026-04-14 15:54:43.387 +08:00 [INF] <-r
2026-04-14 15:57:39.368 +08:00 [INF] ->Trial start from right
2026-04-14 16:01:15.984 +08:00 [INF] ->Trial start from right
2026-04-14 17:18:37.538 +08:00 [INF] <-r
2026-04-14 17:28:00.188 +08:00 [INF] ->Trial start from left
2026-04-14 18:30:18.337 +08:00 [INF] <-r
2026-04-14 18:37:58.017 +08:00 [INF] ->Trial start from right
2026-04-14 18:41:38.351 +08:00 [INF] <-r
2026-04-14 18:41:40.202 +08:00 [INF] ->Trial start from right
2026-04-14 19:01:23.533 +08:00 [INF] <-r
2026-04-14 19:06:52.018 +08:00 [INF] ->Trial start from rig